In [ ]:
import pandas as pd
import json 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Multiple Dataset

In [ ]:
order = [
    #Popular Representation
    'pipe_serialized',
    'token_serialized',
    'space_serialized',
    "centroid_popular",
    # Data Representation
    'csv',
    'tsv',
    'html',
    'markdown',
    'latex',
    'dict',
    'json',
    'xml',
    "centroid_data",
    # Structural Transformations 
    'shuffled_rows',
    'shuffled_cols',
    'transpose',
    "centroid_structural",
    #Schema Definition Types
    'mschema',
    'macschema',
    'ddl',
    "centroid_schema",
     "centroid_all",
]
'''# Top 5
"tsv_csv_space_serialized_ddl_latex",
"tsv_csv_pipe_serialized_space_serialized_transpose",
"xml_dict_json_pipe_serialized_markdown",
"xml_ddl_latex_html_token_serialized",
# Top 5 All
"csv_tsv_pipe_serialized_space_serialized_xml_latex",'''
category = {"Popular Representation": ['pipe_serialized',    'token_serialized', 'space_serialized',"centroid_popular"],
    "Data Representation" : [
            'csv',
            'tsv',
            'html',
            'markdown',
            'latex',
            'dict',
            'json',
            'xml',
            'centroid_data'],
    "Structural Transformations": ['shuffled_rows',
        'shuffled_cols',
        'transpose',
        'centroid_structural'],
    "Schema Definition Types" :['mschema',
    'macschema',
    'ddl',
    'centroid_schema'],
    "All" :['centroid_all'],
    "Top 5": [  "tsv_csv_space_serialized_ddl_latex","tsv_csv_pipe_serialized_space_serialized_transpose","xml_dict_json_pipe_serialized_markdown","xml_ddl_latex_html_token_serialized"],
    "Top 5 All": ["csv_tsv_pipe_serialized_space_serialized_xml_latex"],
}
category_wo_centroid = {"Popular Representation": ['pipe_serialized',    'token_serialized', 'space_serialized'],
    "Data Representation" : [
            'csv',
            'tsv',
            'html',
            'markdown',
            'latex',
            'dict',
            'json',
            'xml',
            ],
    "Structural Transformations": ['shuffled_rows',
        'shuffled_cols',
        'transpose'],
    "Schema Definition Types" :['mschema',
    'macschema',
    'ddl'],
    "Top 5": [   "tsv_csv_space_serialized_ddl_latex","tsv_csv_pipe_serialized_space_serialized_transpose","xml_dict_json_pipe_serialized_markdown","xml_ddl_latex_html_token_serialized"],
     "Top 5 All": ["csv_tsv_pipe_serialized_space_serialized_xml_latex"]
}

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
recall_value = 1

# same color for the three baseline models, separate color for reasonir
shade_levels = {
    "mpnet": 0.55,
    "reasonir": 0.70,
    "bge": 0.60,
    "splade": 0.80,
}

palette = {
    "mpnet": plt.cm.Blues(shade_levels["mpnet"]),
    "reasonir": plt.cm.PuRd(shade_levels["bge"]),
    "bge": plt.cm.YlGn(shade_levels["reasonir"]),
    "splade": plt.cm.OrRd(shade_levels["splade"]),
}
base_models = ["mpnet", "reasonir", "bge", "splade"]
for recall_value in [1]:
    #for dataset in ["WTQ","NQ-Tables","WIKISQL"]:
    for dataset in ["WTQ","WIKISQL","NQ"]:
        df_list = []
        #for model in ["reasonir","splade","mpnet","bge","bm25"]:
        for model in base_models:#"reasonir",,"bge",
            with open(f'./data/retrieval_all/{model}_results/{dataset}/perturbation_results.json') as f:
                data = json.load(f)
            df = pd.DataFrame(data).T
            df = df.reset_index().rename(columns={"index": "Index"})
            df["Representation"] = df["Index"].apply(
                lambda x: x.split("+")[-1].strip()
                ) 
            df["Model"]= [model]*df.shape[0]
            df_list.append(df)
        df = pd.concat(df_list)
        df = df.reset_index(drop=True)
        print(df["Model"].unique())
        #print(df[df[[x for x in df.columns if x!="combo_reps"]].duplicated(keep=False)])

        plt.figure(figsize=(18, 6))

        ax = sns.barplot(
            data=df,
            x="Representation",
            y=f"recall@{recall_value}",
            hue="Model",
            order=order,
            hue_order=base_models,
            palette=palette
        )

        # ---------- category structure ----------
        categories_in_order = [
            "Popular Representation",
            "Data Representation",
            "Structural Transformations",
            "Schema Definition Types",
            "All",
        #    "Top 5",
        #    "Top 5 All",
        ]

        category_sizes = [
            len(category["Popular Representation"]),
            len(category["Data Representation"]),
            len(category["Structural Transformations"]),
            len(category["Schema Definition Types"]),
            len(category["All"]),
        #    len(category["Top 5"]),
        #    len(category["Top 5 All"])
        ]

        # cumulative positions
        starts = np.cumsum([0] + category_sizes[:-1])
        ends = np.cumsum(category_sizes)
        centers = (starts + ends - 1) / 2

        # ---------- vertical separators ----------
        for boundary in ends[:-1]:
            ax.axvline(
                boundary - 0.5,
                color="black",
                linestyle="-",
                linewidth=1.2,
                alpha=0.5
            )

        # ---------- category titles ----------
        ymax = ax.get_ylim()[1]
        for center, label in zip(centers, categories_in_order):
            ax.text(
                center,
                ymax * 1.03,      # push above bars
                label,
                ha="center",
                va="bottom",
                fontsize=11,
                fontweight="bold"
            )

        # ---------- labels & formatting ----------
        plt.ylabel(f"recall@{recall_value}")
        plt.xlabel("Serialization Method")
        #plt.title(f"{dataset} | recall@{recall_value} by Serialization Method", pad=30)
        print(f"{dataset} | recall@{recall_value} by Serialization Method")
        plt.xticks(rotation=45, ha="right")

        # ---------- annotate bars ----------
        for p in ax.patches:
            height = p.get_height()
            if not pd.isna(height):
                ax.annotate(
                    f"{height:.2f}",
                    (p.get_x() + p.get_width() / 2, height),
                    ha="center",
                    va="bottom",
                    fontsize=8,
                    xytext=(0, 3),
                    textcoords="offset points"
                )

        # ---------- legend ----------
        ax.legend(
            title="Model",
            bbox_to_anchor=(1, 1),
            loc="upper left",
            frameon=False
        )

        # extra top space for category labels
        plt.subplots_adjust(top=0.80)
        plt.tight_layout()
        plt.show()


In [ ]:
all_summary = []
subset_order = [ 'pipe_serialized','token_serialized','space_serialized',
                'csv','tsv', 'html','markdown','latex','dict','json','xml',
                'shuffled_rows','shuffled_cols','transpose',
                'mschema','macschema','ddl',
                "centroid_all",
]
for recall_value in [1]:
    for dataset in ["WTQ", "WIKISQL", "NQ"]:
        df_list = []

        for model in base_models:
            with open(f'./data/retrieval_all/{model}_results/{dataset}/perturbation_results.json') as f:
                data = json.load(f)

            temp = pd.DataFrame(data).T.reset_index().rename(columns={"index": "Index"})
            temp["Representation"] = temp["Index"].apply(lambda x: x.split("+")[-1].strip())
            temp["Model"] = model
            temp["Dataset"] = dataset
            df_list.append(temp)

        df = pd.concat(df_list).reset_index(drop=True)
        df = df[df["Representation"].isin(subset_order)].copy()
        metric = f"recall@{recall_value}"
        summary = (
            df.groupby(["Dataset", "Model", "Representation"])[metric]
              .agg(["mean", "std"])
              .reset_index()
        )

        summary["mean_std"] = summary.apply(
            lambda row: f"{row['mean']:.2f}",
            axis=1
        )

        all_summary.append(summary)

final_summary = pd.concat(all_summary, ignore_index=True)

latex_table = (
    final_summary.pivot(
        index=["Dataset", "Model"],
        columns="Representation",
        values="mean_std"
    )
    .reindex(columns=subset_order)
    .reindex(index=pd.MultiIndex.from_product(
        [["WTQ", "WIKISQL", "NQ"], base_models],
        names=["Dataset", "Model"]
    ))
)

print(
    latex_table.to_latex(
        escape=False,
        na_rep="",
        multirow=True,
        caption=f"Results for recall@{recall_value}",
        label=f"tab:recall_{recall_value}_by_dataset_model",
        column_format="ll" + "c" * len(latex_table.columns)
    )
)

In [ ]:
# separate table with best representation per row
mean_table = (
    final_summary.pivot(
        index=["Dataset", "Model"],
        columns="Representation",
        values="mean"
    )
    .reindex(columns=order)
    .reindex(index=pd.MultiIndex.from_product(
        [["WTQ", "WIKISQL", "NQ"], base_models],
        names=["Dataset", "Model"]
    ))
)

max_repr = mean_table.idxmax(axis=1).to_frame(name="Best Representation")

print(max_repr.to_string())

print(
    max_repr.to_latex(
        escape=False,
        multirow=True,
        caption=f"Best representation per dataset-model row for recall@{recall_value}",
        label=f"tab:best_repr_recall_{recall_value}",
        column_format="lll"
    )
)

## Check standard deviation within representation

In [ ]:
# after you build `df = pd.concat(df_list)` for a dataset

metric = f"recall@{recall_value}"
for dataset in ["WTQ","WIKISQL"]:
    df_list = []
    #for model in ["reasonir","splade","mpnet","bge","bm25"]:
    for model in ["reasonir","splade","mpnet","bge",]:
        with open(f'./data/retrieval_all/{model}_results/{dataset}/perturbation_results.json') as f:
            data = json.load(f)
        df = pd.DataFrame(data).T
        df = df.reset_index().rename(columns={"index": "Index"})
        df["Representation"] = df["Index"].apply(
            lambda x: x.split("+")[-1].strip()
            ) 
        df["Model"]= [model]*df.shape[0]
        df_list.append(df)
    df = pd.concat(df_list)
    # keep only the non-centroid reps you care about
    cat_rows = []
    for cat, reps in category_wo_centroid.items():
        tmp = df[df["Representation"].isin(reps)].copy()
        tmp["Category"] = cat
        cat_rows.append(tmp)

    df_cat = pd.concat(cat_rows, ignore_index=True)

    # per-model fluctuation within each category (across reps)
    stats = (
        df_cat
        .groupby(["Model", "Category"], as_index=False)
        .agg(
            n_reps=("Representation", "nunique"),
            mean=(metric, "mean"),
            sd=(metric, "std"),
            min_val=(metric, "min"),
            max_val=(metric, "max"),
        )
    )
    stats["range"] = stats["max_val"] - stats["min_val"]

    # print nicely
    print(f"\n{dataset} | Per-model fluctuation within category (across representations):")
    for model in stats["Model"].unique():
        print(f"\n=== {model} ===")
        print(
            stats[stats["Model"] == model]
            .sort_values("Category")
            [["Category", "n_reps", "mean", "sd", "range", "min_val", "max_val"]]
            .to_string(index=False, float_format=lambda x: f"{x:.4f}")
        )

    # (optional) how much models disagree within each category×rep:
    # average over Index within each Model first (if you have multiple per rep)
    rep_means = (
        df_cat
        .groupby(["Model", "Category", "Representation"], as_index=False)[metric]
        .mean()
    )
    model_disagreement = (
        rep_means
        .groupby(["Category", "Representation"], as_index=False)
        .agg(sd_across_models=(metric, "std"))
    )

    print(f"\n{dataset} | SD across models for each Category × Representation:")
    print(model_disagreement.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

    # all non-centroid reps (union of category_wo_centroid)
    all_reps = sorted({r for reps in category_wo_centroid.values() for r in reps})

    df_all = df[df["Representation"].isin(all_reps)].copy()

    overall = (
        df_all
        .groupby("Model", as_index=False)
        .agg(
            n_reps=("Representation", "nunique"),
            mean=(metric, "mean"),
            sd=(metric, "std"),
            min_val=(metric, "min"),
            max_val=(metric, "max"),
        )
    )
    overall["range"] = overall["max_val"] - overall["min_val"]

    print(f"\n{dataset} | Overall (no category): per-model fluctuation across ALL non-centroid representations")
    print(
        overall[["Model", "n_reps", "mean", "sd", "range", "min_val", "max_val"]]
        .sort_values("Model")
        .to_string(index=False, float_format=lambda x: f"{x:.4f}")
    )
    print("+++++"*100)


# Compare Representation Variation

In [ ]:
category = {
    "Popular Representation": ['pipe_serialized', 'token_serialized', 'space_serialized', "centroid_popular"],
    "Data Representation": ['csv','tsv','html','markdown','latex','dict','json','xml','centroid_data'],
    "Structural Transformations": ['shuffled_rows','shuffled_cols','transpose','centroid_structural'],
    "Schema and Definition Types": ['mschema','macschema','ddl','centroid_schema'],
    "All": ['pipe_serialized', 'token_serialized', 'space_serialized','csv','tsv','html','markdown','latex','dict','json','xml','shuffled_rows','shuffled_cols','transpose','mschema','macschema','ddl','centroid_all']
}

category_wo_centroid = {
    "Popular Representation": ['pipe_serialized','token_serialized','space_serialized'],
    "Data Representation": ['csv','tsv','html','markdown','latex','dict','json','xml'],
    "Structural Transformations": ['shuffled_rows','shuffled_cols','transpose'],
    "Schema and Definition Types": ['mschema','macschema','ddl'],
    "All": ['pipe_serialized', 'token_serialized', 'space_serialized','csv','tsv','html','markdown','latex','dict','json','xml','shuffled_rows','shuffled_cols','transpose','mschema','macschema','ddl'],
}
representation = [
    #Popular Representation
    'pipe_serialized',
    'token_serialized',
    'space_serialized',
    "centroid_popular",
    # Data Representation
    'csv',
    'tsv',
    'html',
    'markdown',
    'latex',
    'dict',
    'json',
    'xml',
    "centroid_data",
    # Structural Transformations 
    'shuffled_rows',
    'shuffled_cols',
    'transpose',
    "centroid_structural",
    #Schema and Definition Types
    'mschema',
    'macschema',
    'ddl',
    "centroid_schema",
     "centroid_all",
]
df_list = []
for dataset in ["WTQ","WIKISQL","NQ"]:
    #for model in ["reasonir","splade","mpnet","bge","bm25"]:
    for model in ["splade","mpnet","bge","reasonir",]:#"reasonir",
        with open(f'./data/retrieval_all/{model}_results/{dataset}/perturbation_results.json') as f:
            data = json.load(f)
        df = pd.DataFrame(data).T
        df = df.reset_index().rename(columns={"index": "Index"})
        df["Representation"] = df["Index"].apply(
            lambda x: x.split("+")[-1].strip()
            ) 
        df = df[df["Representation"].isin(representation)]
        df["Model"]= [model]*df.shape[0]
        df["Dataset"]= [dataset]*df.shape[0]
        df_list.append(df)
df = pd.concat(df_list)


def get_category(rep):
    if rep == "centroid_all":
        return "All"
    for cat, reps in category.items():
        if rep in reps and cat != "All":
            return cat
    return None
df["Category"] = df["Representation"].apply(get_category)


In [ ]:
metric = "recall@1"
df["IsCentroid"] = df["Representation"].apply(lambda x: True if 'centroid' in x else False)
df_wo_centroid = df[~df["IsCentroid"]].copy()
df_wo_centroid["rank"] = (
    df_wo_centroid.groupby(["Model", "Dataset"])[metric]
      .rank(ascending=False, method="average")
)
df_wo_centroid =df_wo_centroid.sort_values(["Model", "Dataset","rank"])

## Variation of the performance

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

metric = 'recall@1'
shade_levels = {
    "mpnet": 0.55,
    "reasonir": 0.70,
    "bge": 0.60,
    "splade": 0.80,
}

palette = {
    "mpnet": plt.cm.Blues(shade_levels["mpnet"]),
    "reasonir": plt.cm.PuRd(shade_levels["bge"]),
    "bge": plt.cm.Greens(shade_levels["reasonir"]),
    "splade": plt.cm.OrRd(shade_levels["splade"]),
}
base_models = ["mpnet", "reasonir", "bge", "splade"]
datasets = ["WTQ", "WIKISQL", "NQ"]
# -----------------------------
# 1) Keep only non-centroid rows
# -----------------------------
df = df_wo_centroid.copy()

# If df_wo_centroid already excludes centroid, this is harmless.
if 'IsCentroid' in df.columns:
    df = df[df['IsCentroid'] == False].copy()

# Optional: exclude combo rows if they exist in this df
# (You said this df is "wo centroid", but combos can still appear)
if 'combo_reps' in df.columns:
    df = df[df['combo_reps'].isna()].copy()

# -----------------------------
# 2) Compute variation per Dataset x Model
# -----------------------------
variation = (
    df.groupby(['Dataset', 'Model'])[metric]
      .agg(std='std', min_val='min', max_val='max', mean='mean', n='count')
      .reset_index()
)

variation['range'] = variation['max_val'] - variation['min_val']  # max - min

print("Per Dataset x Model variation:")
print(variation.sort_values(['Dataset', 'Model']).to_string(index=False))

# -----------------------------
# 3) Dataset-level summary
#    (average variation across models)
# -----------------------------
dataset_summary = (
    variation.groupby('Dataset')[['std', 'range']]
             .mean()
             .reset_index()
             .rename(columns={
                 'std': 'avg_std_across_models',
                 'range': 'avg_range_across_models'
             })
)

print("\nDataset-level summary:")
print(dataset_summary.sort_values('avg_range_across_models', ascending=False).to_string(index=False))

# -----------------------------
# 4) Plot standard deviation
# -----------------------------
std_pivot = variation.pivot(index='Dataset', columns='Model', values='std')
std_pivot = std_pivot.reindex(index=datasets,columns=base_models)
ax = std_pivot.plot(kind='bar', figsize=(8, 4), rot=0,color = palette)
ax.set_title(f'Std Dev of {metric} Across Representations')
ax.set_xlabel('Dataset')
ax.set_ylabel('Standard Deviation')
ax.legend(title='Model', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

# -----------------------------
# 5) Plot range (max - min)
# -----------------------------
range_pivot = variation.pivot(index='Dataset', columns='Model', values='range')
range_pivot = range_pivot.reindex(index=datasets,columns=base_models)

ax = range_pivot.plot(kind='bar', figsize=(8, 4), rot=0,color = palette)
ax.set_title('Range of recall@1 Across Representations (Max - Min)')
ax.set_xlabel('Dataset')
ax.set_ylabel('Range')
ax.legend(title='Model', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

# -----------------------------
# 6) Single-number view for your claim
# -----------------------------
print("\nRanking by avg_range_across_models (higher = more variation):")
print(dataset_summary.sort_values('avg_range_across_models', ascending=False).to_string(index=False))


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

std_pivot = variation.pivot(index='Dataset', columns='Model', values='std')
std_pivot = std_pivot.reindex(index=datasets, columns=base_models)

range_pivot = variation.pivot(index='Dataset', columns='Model', values='range')
range_pivot = range_pivot.reindex(index=datasets, columns=base_models)

x = np.arange(len(datasets))
width = 0.18
offsets = np.linspace(-1.5 * width, 1.5 * width, len(base_models))

fig, ax = plt.subplots(figsize=(10, 5))

for i, model in enumerate(base_models):
    ax.bar(
        x + offsets[i],
        std_pivot[model],
        width=width,
        color=palette[model],
        label=f'{model} std'
    )
    ax.bar(
        x + offsets[i],
        -range_pivot[model],
        width=width,
        color=palette[model],
        label=f'{model} range' if i == 0 else None
    )

ax.axhline(0, color='black', linewidth=1)
ax.set_ylim(-ymax * 1.15, ymax * 1.15-0.1)
ax.set_xticks(x)
ax.set_xticklabels(datasets)
ax.set_ylabel(' Bottom: Range  |  Top: Std Dev')
ax.set_title(f'Standard Deviation and Bottom = Range for {metric}')

# Cleaner legend: one for models, one note in caption/title is usually better
handles, labels = ax.get_legend_handles_labels()
unique = dict(zip(labels, handles))
unique = {d:unique[d+" std"] for d in base_models}
ax.legend(unique.values(), unique.keys(), title='Model', bbox_to_anchor=(1.02, 1), loc='upper left')


plt.tight_layout()
plt.show()

## Best representation by model?

In [ ]:
import matplotlib.pyplot as plt

metric = "recall@1"

avg = (
    df_wo_centroid.groupby(["Model", "Representation"], as_index=False)[metric]
      .mean()
)
avg["rank"] = (
    avg.groupby("Model")[metric]
       .rank(method="first", ascending=False)
)
avg = avg.sort_values(["Model", "rank"])


models = avg["Model"].unique()
n_models = len(models)

fig, axes = plt.subplots(
    n_models, 1,
    figsize=(10, 3 * n_models),
    sharex=False
)

if n_models == 1:
    axes = [axes]

for ax, model in zip(axes, models):
    sub = avg[avg["Model"] == model]

    ax.barh(
        sub["Representation"],
        sub[metric]
    )

    ax.invert_yaxis()  # best at top
    ax.set_title(model)
    ax.set_xlabel(metric)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

metric = "recall@1"

avg = (
    df_wo_centroid.groupby(["Representation"], as_index=False)[metric]
      .mean()
)
avg["rank"] = (
    avg[metric]
       .rank(method="first", ascending=False)
)
avg = avg.sort_values(["rank"])


fig, axes = plt.subplots(
    1, 1,
    figsize=(10, 3),
)

axes = [axes]

for ax in axes:
    sub = avg

    ax.barh(
        sub["Representation"],
        sub[metric]
    )

    ax.invert_yaxis()  # best at top
    ax.set_title("All Average")
    ax.set_xlabel(metric)

plt.tight_layout()
plt.show()


# By how much does the rank vary when I change representation?

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt



rank_stats = df_wo_centroid["rank"].agg(["min", "max", "std"])
rank_stats

# Which representations are consistently performing better?

In [ ]:
print(df["Dataset"].unique())
print(df["Model"].unique())

In [ ]:
rep_rank = (
    df_wo_centroid
    .groupby("Representation")["rank"]
    .agg(["mean", "std"])
    .reset_index()
    .sort_values("mean")
)

plt.figure(figsize=(6, 4))
plt.errorbar(
    rep_rank["mean"],
    rep_rank["Representation"],
    xerr=rep_rank["std"],
    fmt="o"
)

plt.xlabel("Mean Rank (lower = better)")
plt.ylabel("")
plt.title("Which Table Representations Rank Best on Average")
plt.tight_layout()
plt.show()


In [ ]:
heat_df = (
    df_wo_centroid
    .pivot_table(
        index="Representation",
        columns=["Model", "Dataset"],
        values="rank"
    )
)

plt.figure(figsize=(8, 6))
sns.heatmap(
    heat_df,
    cmap="viridis_r",
    annot=False,
    cbar_kws={"label": "Rank (lower = better)"}
)

plt.title("Rank of Each Representation Across Models and Datasets")
plt.ylabel("")
plt.xlabel("")
plt.tight_layout()
plt.show()

